# Prever preço de diamantes com Árvore de Decisão

O conjunto de dados "**Diamonds**" do *seaborn* é frequentemente usado para estudos de regressão e contém variáveis categóricas e numéricas.<br>
Esse conjunto de dados tem informações sobre diamantes, incluindo o preço (que será a variável alvo para regressão), o quilate, a cor, a claridade, entre outras características.<br>
**Descrição das Colunas**:

- **carat**: Peso do diamante.
- **cut**: Qualidade do corte do diamante (Fair, Good, Very Good, Premium, Ideal).
- **color**: Cor do diamante, de J (menos valioso) a D (mais valioso).
- **clarity**: Claridade do diamante (I1, SI2, SI1, VS2, VS1, VVS2, VVS1, IF).
- **depth**: Profundidade total do diamante como uma porcentagem da média do diâmetro.
- **table**: Largura da mesa do diamante como uma porcentagem da média do diâmetro.
- **price**: Preço do diamante (variável alvo).
- **x**: Comprimento em milímetros.
- **y**: Largura em milímetros.
- **z**: Profundidade em milímetros.


## import Bibliotecas

In [ ]:
# from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from ydata_profiling import ProfileReport


## Fixar a seed para reprodutibilidade

In [ ]:
np.random.seed(42)

## Carregar os dados

In [ ]:
df = sns.load_dataset('diamonds')

## Informações básicas do DataFrame

In [ ]:
print(df.info())

## Quantidade de linhas e colunas

In [ ]:
df.shape

## Estatísticas descritivas

In [ ]:
display(df.describe())

In [ ]:
display(df.describe().T)

# Análise Exploratória de Dados (EDA)

## Distribuição do target (preço)


In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(df['price'], bins=30, kde=True)
plt.title('Distribuição dos Preços')
plt.xlabel('Price')
plt.ylabel('Frequência')
plt.show()


**Comentário:**  
  
A distribuição de `price` é fortemente enviesada à direita (long tail).  
Recomenda-se analisar `log1p(price)` para estabilizar variância e melhorar ajuste de modelos lineares

In [ ]:
# Transformação do target
_df = df.copy()
_df['price_log'] = np.log1p(_df['price'])
plt.figure(figsize=(8,6))
sns.histplot(_df['price_log'], bins=30, kde=True, color='C1')
plt.title('Distribuição de log1p(price)')
plt.show()

## Análise das variáveis

### carat
- Peso do diamante  

#### tipo de dado

In [ ]:
print(df['carat'].dtype)

#### valores unicos

In [ ]:
print(df['carat'].nunique())


#### distribuição

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(data=df, x='carat')
plt.title('Contagem dos valores na coluna carat')
plt.show()


#### histograma

In [ ]:
df['carat'].hist(bins=30)
plt.title('Histograma da coluna carat')
plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.show()

#### boxplot

In [ ]:
sns.boxplot(data=df, x='carat')
plt.title('Boxplot da coluna carat')
plt.show()

**Comentário:**  
`carat` possui distribuição assimétrica com alguns outliers.  
Recomenda-se criar `carat_log = log1p(carat)` para modelagem linear e `carat_bin` para análises descritivas.

In [ ]:
# Transformações em carat

df['carat_log'] = np.log1p(df['carat'])
plt.figure(figsize=(8,6))
sns.histplot(df['carat_log'], bins=30, kde=True, color='C1')
plt.title('Distribuição de log1p(carat)')
plt.show()

In [ ]:
# exemplo de binning simples
df['carat_bin'] = pd.cut(df['carat'], bins=[0,0.5,1,1.5,2,10], labels=['<=0.5','0.5-1','1-1.5','1.5-2','>2'])
df['carat_bin'].value_counts()
plt.figure(figsize=(8,6))
sns.countplot(data=df, x='carat_bin', order=['<=0.5','0.5-1','1-1.5','1.5-2','>2'])
plt.title('Contagem dos bins de carat')
plt.xlabel('Faixa de Carat')
plt.ylabel('Frequência')
plt.show()

### cut

- Qualidade do corte do diamante (Fair, Good, Very Good, Premium, Ideal).

#### tipo de dado

In [ ]:
print(df['cut'].dtype)

#### valores unicos

In [ ]:
print(df['cut'].nunique())


#### distribuição

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(data=df, x='cut')
plt.title('Contagem dos valores na coluna cut')
plt.show()


#### histograma

In [ ]:
df['cut'].hist(bins=30)
plt.title('Histograma da coluna cut')
plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.show()

#### PIZZA

In [ ]:
conta = df['cut'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(conta, labels=conta.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribuição da coluna cut')
plt.show()


### color

- Cor do diamante, de J (menos valioso) a D (mais valioso).


#### tipo de dado

In [ ]:
print(df['color'].dtype)

#### valores unicos

In [ ]:
print(df['color'].nunique())


#### distribuição

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(data=df, x='color')
plt.title('Contagem dos valores na coluna color')
plt.show()


#### histograma

In [ ]:
df['color'].value_counts().sort_index().plot(kind='bar')
#df['color'].hist(bins=30)
plt.title('Histograma da coluna color')
plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.show()

In [ ]:
# Gráfico em percentual por categoria (ordenado)
counts = df['color'].value_counts().sort_index()
percent = counts / counts.sum() * 100
fig, ax = plt.subplots(figsize=(10, 6))
percent.plot(kind='bar', color='C2', ax=ax)
ax.set_title('Percentual por categoria - color')
ax.set_xlabel('color')
ax.set_ylabel('Percentual (%)')
# Anotar valores percentuais acima das barras
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f"{height:.1f}%", (p.get_x() + p.get_width() / 2, height),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

#### PIZZA

In [ ]:
conta = df['color'].value_counts().sort_index()
plt.figure(figsize=(8, 8))
plt.pie(conta, labels=conta.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribuição da coluna conta')
plt.show()


### clarity

- Claridade do diamante (I1, SI2, SI1, VS2, VS1, VVS2, VVS1, IF).  


#### tipo de dado

In [ ]:
print(df['clarity'].dtype)

#### valores unicos

In [ ]:
print(df['clarity'].nunique())


#### distribuição

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(data=df, x='clarity')
plt.title('Contagem dos valores na coluna clarity')
plt.show()


In [ ]:
# Gráfico em percentual por categoria (ordenado)
counts = df['clarity'].value_counts().sort_index()
percent = counts / counts.sum() * 100
fig, ax = plt.subplots(figsize=(10, 6))
percent.plot(kind='bar', color='C2', ax=ax)
ax.set_title('Percentual por categoria - clarity')
ax.set_xlabel('clarity')
ax.set_ylabel('Percentual (%)')
# Anotar valores percentuais acima das barras
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f"{height:.1f}%", (p.get_x() + p.get_width() / 2, height),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

#### PIZZA

In [ ]:
conta = df['clarity'].value_counts().sort_index()
plt.figure(figsize=(8, 8))
plt.pie(conta, labels=conta.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribuição da coluna conta')
plt.show()


### depth

- Profundidade total do diamante como uma porcentagem da média do diâmetro.  


#### tipo de dado

In [ ]:
print(df['depth'].dtype)

#### valores unicos

In [ ]:
print(df['depth'].nunique())


#### distribuição

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(data=df, x='depth')
plt.title('Contagem dos valores na coluna depth')
plt.show()


#### histograma

In [ ]:
df['depth'].hist(bins=30)
plt.title('Histograma da coluna depth')
plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.show()

#### boxplot

In [ ]:
sns.boxplot(data=df, x='depth')
plt.title('Boxplot da coluna depth')
plt.show()

### table

- Largura da mesa do diamante como uma porcentagem da média do diâmetro.  

#### tipo de dado

In [ ]:
print(df['table'].dtype)

#### valores unicos

In [ ]:
print(df['table'].nunique())


#### distribuição

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(data=df, x='table')
plt.title('Contagem dos valores na coluna table')
plt.show()


#### histograma

In [ ]:
df['table'].hist(bins=30)
plt.title('Histograma da coluna table')
plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.show()

#### boxplot

In [ ]:
sns.boxplot(data=df, x='table')
plt.title('Boxplot da coluna table')
plt.show()

### x

- Comprimento em milímetros.  


#### tipo de dado

In [ ]:
print(df['x'].dtype)

#### valores unicos

In [ ]:
print(df['x'].nunique())


#### distribuição

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(data=df, x='x')
plt.title('Contagem dos valores na coluna x')
plt.show()


#### histograma

In [ ]:
df['x'].hist(bins=30)
plt.title('Histograma da coluna x')
plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.show()

#### boxplot

In [ ]:
sns.boxplot(data=df, x='x')
plt.title('Boxplot da coluna x')
plt.show()

### y

- Largura em milímetros.  


#### tipo de dado

In [ ]:
print(df['y'].dtype)

#### valores unicos

In [ ]:
print(df['y'].nunique())


#### distribuição

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(data=df, x='y')
plt.title('Contagem dos valores na coluna y')
plt.show()


#### histograma

In [ ]:
df['y'].hist(bins=30)
plt.title('Histograma da coluna y')
plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.show()

#### boxplot

In [ ]:
sns.boxplot(data=df, x='y')
plt.title('Boxplot da coluna y')
plt.show()

### z

- Profundidade em milímetros.


#### tipo de dado

In [ ]:
print(df['z'].dtype)

#### valores unicos

In [ ]:
print(df['z'].nunique())


#### distribuição

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(data=df, x='z')
plt.title('Contagem dos valores na coluna z')
plt.show()


#### histograma

In [ ]:
df['z'].hist(bins=30)
plt.title('Histograma da coluna z')
plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.show()

#### boxplot

In [ ]:
sns.boxplot(data=df, x='z')
plt.title('Boxplot da coluna z')
plt.show()

**Comentário:** 

- verificar e remover linhas com dimensões zero (x==0 | y==0 | z==0)
- criar `volume = x*y*z` e analisar correlação com `price`. 
- Se houver muitas linhas inválidas documentar a origem.

## Feature Engeniering

### Volume

In [ ]:
# Limpeza e feature derivada: volume
invalid = (df['x'] == 0) | (df['y'] == 0) | (df['z'] == 0)
print('Linhas com dimensões inválidas:', invalid.sum())
df = df[~invalid].copy()
df['volume'] = df['x'] * df['y'] * df['z']

In [ ]:
df['volume'].hist(bins=30)
plt.title('Histograma da coluna volume')
plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.show()

### volume_log

In [ ]:
df['volume_log'] = np.log1p(df['volume'])
print('Exemplo: correlação volume vs price:', df[['volume','price']].corr().iloc[0,1])

In [ ]:
df['volume_log'].hist(bins=30)
plt.title('Histograma da coluna volume_log')
plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.show()

## Matriz de correlação

In [ ]:
plt.figure(figsize=(12, 8))
corr_matrix = df.corr(numeric_only=True)
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Matriz de Correlação')
plt.show()

**Comentário:** 

- calcular VIF para detectar multicolinearidade entre variáveis numéricas (ex.: x,y,z).

## VIF

In [ ]:
X = df[['carat','carat_log','depth','table','x','y','z','volume', 'volume_log']].dropna()
vif_data = pd.DataFrame({'feature': X.columns, 'vif': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
display(vif_data)


In [ ]:
vif_plot = vif_data.sort_values('vif', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=vif_plot, x='feature', y='vif', palette='viridis')
plt.title('VIF por variável')
plt.xlabel('Variável')
plt.ylabel('VIF')
plt.ylim(0, vif_plot['vif'].max() * 1.1)

for i, v in enumerate(vif_plot['vif']):
    plt.text(i, v + vif_plot['vif'].max() * 0.02, f'{v:.1f}', ha='center')

plt.tight_layout()
plt.show()

Analise

O resultado mostra multicolinearidade muito alta entre as variáveis:

- VIF > 10 já indica problema sério; aqui todos os valores estão muito acima disso.
- `x`, `y` e `z` têm VIF extremamente altos, o que é esperado porque são medidas de dimensão muito correlacionadas.
- `volume` também apresenta VIF alto, pois é derivado de `x * y * z` e, portanto, é redundante em relação a essas variáveis.
- `carat` tem VIF elevado porque está fortemente correlacionado com as dimensões do diamante.
- `depth` e `table` também têm VIF alto, indicando que há multicolinearidade mesmo entre variáveis independentes.

Conclusão:
- o modelo linear com essas variáveis terá instabilidade nos coeficientes;
- é recomendável remover ou agrupar variáveis redundantes (por exemplo, manter só `carat` ou só `volume`/`x,y,z`);
- outra alternativa é usar regularização ou redução de dimensionalidade.

## Scatter plots entre features numéricas + target

In [ ]:
colunas = ['carat', 'carat_log', 'depth', 'table', 'x', 'y', 'z', 'volume', 'volume_log', 'price']
sns.pairplot(df[colunas])
plt.show()

# Relatório automático (ydata-profiling)

In [ ]:
profile = ProfileReport(df, title='Diamonds profile', explorative=True)
profile.to_file('diamonds_profile.html')
print('Relatório salvo em diamonds_profile.html')
